# Cross-Cutting: Global Health Resilience Index (GHRI)
Novel composite index with PCA-weighted geometric aggregation. 5 pillars: Disease Burden, Risk Factor Control, Health System Capacity, Socioeconomic Foundation, Resilience & Equity.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from hdi.config import *
from hdi.data.loaders import load_panel
from hdi.features.indices import compute_ghri, bootstrap_ghri_ci
from hdi.visualization.charts import plot_heatmap, plot_bump_chart, plot_scatter
from hdi.visualization.interactive import interactive_ghri_radar

panel = load_panel()
print(f"Panel shape: {panel.shape}")
panel.head()

## 1. GHRI Construction

In [ ]:
# Compute the Global Health Resilience Index
ghri_df = compute_ghri(panel)

# Display top 20 countries by GHRI score
top20 = ghri_df.sort_values("ghri_score", ascending=False).head(20)
print("Top 20 Countries by Global Health Resilience Index (GHRI)")
print("=" * 60)
display(top20[["country", "ghri_score", "ghri_rank",
               "disease_burden", "risk_factor_control",
               "health_system_capacity", "socioeconomic_foundation",
               "resilience_equity"]].reset_index(drop=True))

## 2. Pillar Score Heatmap

In [ ]:
# Heatmap of top 30 countries across 5 GHRI pillars
pillar_cols = [
    "disease_burden", "risk_factor_control",
    "health_system_capacity", "socioeconomic_foundation",
    "resilience_equity"
]
pillar_labels = [
    "Disease Burden", "Risk Factor Control",
    "Health System Capacity", "Socioeconomic Foundation",
    "Resilience & Equity"
]

top30 = ghri_df.sort_values("ghri_score", ascending=False).head(30)
heatmap_data = top30.set_index("country")[pillar_cols]
heatmap_data.columns = pillar_labels

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".2f",
    cmap="YlGnBu",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Pillar Score (0-1)"}
)
ax.set_title("GHRI Pillar Scores: Top 30 Countries", fontsize=14, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap confidence intervals for GHRI ranks
ci_df = bootstrap_ghri_ci(panel, n_boot=1000, seed=42)

# Display rank uncertainty for top 30 countries
top30_ci = ci_df.sort_values("ghri_rank_median").head(30)
print("GHRI Rank Uncertainty (95% Bootstrap CI)")
print("=" * 60)
display(top30_ci[["country", "ghri_rank_median",
                   "ghri_rank_ci_lower", "ghri_rank_ci_upper",
                   "ghri_score_mean", "ghri_score_std"]].reset_index(drop=True))

# Visualize rank uncertainty as error bars
fig, ax = plt.subplots(figsize=(10, 10))
y_pos = range(len(top30_ci))
ax.barh(y_pos, top30_ci["ghri_score_mean"], xerr=[
    top30_ci["ghri_score_mean"] - top30_ci["ghri_score_ci_lower"],
    top30_ci["ghri_score_ci_upper"] - top30_ci["ghri_score_mean"]
], color="steelblue", alpha=0.8, capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(top30_ci["country"])
ax.set_xlabel("GHRI Score")
ax.set_title("GHRI Score with 95% Bootstrap CI (Top 30)", fontsize=13, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Interactive Radar Comparison

In [ ]:
# Interactive radar chart comparing country groups across GHRI pillars
country_groups = {
    "G7": ["United States", "United Kingdom", "Germany", "France", "Japan", "Canada", "Italy"],
    "BRICS": ["Brazil", "Russia", "India", "China", "South Africa"],
    "Nordic": ["Norway", "Sweden", "Denmark", "Finland", "Iceland"],
    "ASEAN-5": ["Indonesia", "Thailand", "Vietnam", "Philippines", "Malaysia"],
}

fig = interactive_ghri_radar(ghri_df, country_groups=country_groups)
fig.show()

## 5. Temporal GHRI Evolution

In [ ]:
# Compute GHRI for multiple years and show evolution
years = sorted(panel["year"].unique())
ghri_temporal = []

for yr in years:
    panel_yr = panel[panel["year"] == yr]
    if len(panel_yr) > 20:  # skip years with too few observations
        ghri_yr = compute_ghri(panel_yr)
        ghri_yr["year"] = yr
        ghri_temporal.append(ghri_yr)

ghri_temporal_df = pd.concat(ghri_temporal, ignore_index=True)

# Bump chart: rank evolution for selected countries
highlight_countries = [
    "United States", "China", "Germany", "Japan",
    "Brazil", "India", "Nigeria", "Norway",
    "South Korea", "United Kingdom"
]
temporal_subset = ghri_temporal_df[ghri_temporal_df["country"].isin(highlight_countries)]

fig, ax = plt.subplots(figsize=(14, 8))
for country in highlight_countries:
    cdata = temporal_subset[temporal_subset["country"] == country].sort_values("year")
    ax.plot(cdata["year"], cdata["ghri_rank"], marker="o", label=country, linewidth=2)
    # Label last point
    if len(cdata) > 0:
        last = cdata.iloc[-1]
        ax.annotate(country, (last["year"], last["ghri_rank"]),
                    textcoords="offset points", xytext=(8, 0), fontsize=8)

ax.invert_yaxis()
ax.set_xlabel("Year")
ax.set_ylabel("GHRI Rank")
ax.set_title("Temporal GHRI Rank Evolution (Bump Chart)", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.15, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 6. GHRI vs HDI Correlation

In [ ]:
# Scatter plot: GHRI vs HDI with correlation
merged = ghri_df.merge(
    panel.groupby("country")["hdi"].mean().reset_index(),
    on="country", how="inner"
)

corr = merged["ghri_score"].corr(merged["hdi"])
rank_corr = merged["ghri_score"].corr(merged["hdi"], method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(merged["hdi"], merged["ghri_score"], alpha=0.6, edgecolors="k", linewidth=0.5)

# Regression line
z = np.polyfit(merged["hdi"], merged["ghri_score"], 1)
p = np.poly1d(z)
x_line = np.linspace(merged["hdi"].min(), merged["hdi"].max(), 100)
ax.plot(x_line, p(x_line), "r--", linewidth=2, label=f"OLS fit")

# Annotate outliers
residuals = merged["ghri_score"] - p(merged["hdi"])
outliers = merged[residuals.abs() > residuals.std() * 1.5]
for _, row in outliers.iterrows():
    ax.annotate(row["country"], (row["hdi"], row["ghri_score"]),
                fontsize=7, alpha=0.8)

ax.set_xlabel("Human Development Index (HDI)", fontsize=12)
ax.set_ylabel("Global Health Resilience Index (GHRI)", fontsize=12)
ax.set_title(f"GHRI vs HDI (Pearson r={corr:.3f}, Spearman rho={rank_corr:.3f})",
             fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nPearson correlation:  {corr:.4f}")
print(f"Spearman correlation: {rank_corr:.4f}")